# 8) M1.3 NumPy, SciPy and Newton's method — Exercises

**Goals:** vectorised energy calculations with broadcasting, a reusable Newton solver for one equation and for systems, a numerical Jacobian to check hand derivations, the loadability limit of a line, and a radial feeder with constant-power loads.

Units: V, A, W, Ω for the circuits; MW, MWh, €/MWh for the energy calculations.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import math

import numpy as np
from scipy import optimize

### Warm-ups

**1. Energy and capacity factors, vectorised**

`availability` has shape (hours, generators) with values in [0, 1]; `p_nom_MW` has shape (generators,). Return the available energy (MWh) and the capacity factor per generator, without any Python loop.

In [ ]:
def energy_and_capacity_factor(availability: np.ndarray, p_nom_MW: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Available energy (MWh, hourly steps) and capacity factor per generator."""
    ...  # YOUR CODE HERE

In [ ]:
avail = np.array([[0.5, 1.0], [0.0, 1.0], [1.0, 0.5]])
energy, cf = energy_and_capacity_factor(avail, np.array([100.0, 40.0]))
assert np.allclose(energy, [150.0, 100.0]), "MW x hours, summed down the hours"
assert np.allclose(cf, [0.5, 2.5 / 3])
assert energy.shape == (2,), "one value per generator"

**2. Total generation cost**

`dispatch_MW` has shape (hours, generators), `marginal_cost` shape (generators,) in €/MWh. Return the total cost in € for hourly steps — as a single matrix–vector product.

In [ ]:
def total_cost_EUR(dispatch_MW: np.ndarray, marginal_cost: np.ndarray) -> float:
    """Total generation cost (EUR) of an hourly dispatch."""
    ...  # YOUR CODE HERE

In [ ]:
d = np.array([[800.0, 200.0], [500.0, 0.0]])
assert math.isclose(total_cost_EUR(d, np.array([30.0, 80.0])), 800 * 30 + 200 * 80 + 500 * 30)

### Core

**3. A reusable Newton solver**

Write `newton(f, df, x0, tol, max_iter)`. It returns the root and the list of iterates (starting with `x0`). Stop when |f(x)| < `tol`. Raise `RuntimeError` if the derivative is zero or if it has not converged after `max_iter` iterations — a solver must never silently return a non-solution.

In [ ]:
def newton(f, df, x0: float, tol: float = 1e-9, max_iter: int = 50) -> tuple[float, list[float]]:
    """Solve f(x) = 0 with Newton's method. Returns (root, iterates)."""
    ...  # YOUR CODE HERE

In [ ]:
root, hist = newton(lambda x: x**2 - 2, lambda x: 2 * x, 1.0)
assert math.isclose(root, math.sqrt(2), rel_tol=1e-9), "|f| < 1e-9 gives about 9 correct digits"
assert len(hist) <= 7, "quadratic convergence: only a handful of iterations"
assert hist[0] == 1.0, "history starts with x0"

try:
    newton(lambda x: x**2 + 1, lambda x: 2 * x, 0.5, max_iter=30)   # no real root
    raise AssertionError("must raise RuntimeError when there is no root")
except RuntimeError:
    pass

**4. How much power can a line deliver?**

A source at `Vs` feeds a constant-power load `P` through resistance `R` (lecture: V(Vs − V)/R = P). Write `max_transfer_W` and `load_voltage`: the high-voltage solution, found with *your* `newton` from a flat start. If P is above the maximum transferable power, raise `ValueError` before trying to solve — check the physics first.

In [ ]:
def max_transfer_W(Vs: float, R: float) -> float:
    """Maximum power (W) a source at Vs can deliver through resistance R."""
    ...  # YOUR CODE HERE


def load_voltage(P: float, Vs: float, R: float) -> float:
    """High-voltage solution of V (Vs - V) / R = P, found with Newton from a flat start."""
    ...  # YOUR CODE HERE

In [ ]:
assert math.isclose(max_transfer_W(100, 1), 2500)
V = load_voltage(1000, 100, 1)
assert math.isclose(V, (100 + math.sqrt(100**2 - 4000)) / 2, rel_tol=1e-9), "the high-voltage root"
assert math.isclose(V * (100 - V), 1000, rel_tol=1e-9), "the load really gets 1,000 W"
try:
    load_voltage(2600, 100, 1)
    raise AssertionError("must raise ValueError above the maximum transfer")
except ValueError:
    pass
for P in [500, 1500, 2400, 2499]:
    print(f"P = {P:5d} W -> V = {load_voltage(P, 100, 1):6.2f} V")

**5. Newton for systems**

Write `newton_system(F, J, x0, tol, max_iter)` for F: ℝⁿ → ℝⁿ with Jacobian J. Each step solves J(x) Δx = −F(x) with `np.linalg.solve` (never invert J). Return the solution and the number of iterations; raise `RuntimeError` without convergence. Test it on the lecture's two-load network.

In [ ]:
def newton_system(F, J, x0, tol: float = 1e-9, max_iter: int = 50) -> tuple[np.ndarray, int]:
    """Solve F(x) = 0 for a vector x with Newton's method. Returns (solution, iterations)."""
    ...  # YOUR CODE HERE

In [ ]:
V0, P1, P2 = 100.0, 800.0, 600.0
F2 = lambda v: np.array([v[0] * (V0 - 2 * v[0] + v[1]) - P1, v[1] * (v[0] - v[1]) - P2])
J2 = lambda v: np.array([[V0 - 4 * v[0] + v[1], v[0]], [v[1], v[0] - 2 * v[1]]])

v, iters = newton_system(F2, J2, [V0, V0])
assert np.allclose(v, optimize.root(F2, [V0, V0], jac=J2).x), "must match scipy"
assert iters <= 6
losses = (V0 - v[0]) ** 2 + (v[0] - v[1]) ** 2
assert math.isclose(V0 * (V0 - v[0]), P1 + P2 + losses), "slack supplies loads + losses"
print(f"V = {v.round(3)} V after {iters} iterations")

**6. Check your Jacobian numerically**

Hand-derived Jacobians are a classic source of bugs. Write `numerical_jacobian(F, x, h)` with **central differences**: column j is (F(x + h·eⱼ) − F(x − h·eⱼ)) / (2h). Use it to check `J2` at several points, then show that Newton still converges with it.

In [ ]:
def numerical_jacobian(F, x, h: float = 1e-6) -> np.ndarray:
    """Central-difference approximation of the Jacobian of F at x."""
    ...  # YOUR CODE HERE

In [ ]:
for point in ([100.0, 100.0], [90.0, 80.0], [82.0, 74.0]):
    assert np.allclose(numerical_jacobian(F2, point), J2(point), atol=1e-4), f"Jacobian differs at {point}"

v_num, _ = newton_system(F2, lambda x: numerical_jacobian(F2, x), [V0, V0])
assert np.allclose(v_num, v)
print("analytic and numerical Jacobians agree; Newton converges with both")

**7. Interpretation**

In task 4, compute the derivative f′(V) = (Vs − 2V)/R at the solution for P = 500, 2,400 and 2,499 W. What happens to it as P approaches the maximum transfer, why does that make Newton's steps unreliable, and what does it mean physically for a grid operator?

*Your answer here.*

### Challenge

**8. A radial distribution feeder**

A substation at `Vs` feeds `n` buses in a chain (bus 1 … n), each segment with resistance `R`, each bus drawing constant power `P_each`. With branch current Iₖ = (Vₖ₋₁ − Vₖ)/R (V₀ = Vs) and I_{n+1} = 0, the mismatch at bus k is fₖ = Vₖ (Iₖ − Iₖ₊₁) − P_each.

Write `feeder_mismatch`, `feeder_jacobian` (derive it by hand — the matrix is tridiagonal; check it with task 6) and `solve_feeder`, which returns the bus voltages from a flat start. Then report the lowest voltage and the losses for 10 buses × 100 W, Vs = 100 V, R = 0.2 Ω per segment.

In [ ]:
def feeder_mismatch(v, Vs: float, R: float, P_each: float) -> np.ndarray:
    """Power mismatch (W) at buses 1..n of a radial feeder."""
    ...  # YOUR CODE HERE


def feeder_jacobian(v, Vs: float, R: float) -> np.ndarray:
    """Jacobian of feeder_mismatch with respect to the bus voltages."""
    ...  # YOUR CODE HERE


def solve_feeder(n: int, Vs: float, R: float, P_each: float) -> np.ndarray:
    """Bus voltages (V) of the feeder, Newton from a flat start."""
    ...  # YOUR CODE HERE

In [ ]:
n, Vs_f, R_f, P_f = 10, 100.0, 0.2, 100.0
test_point = np.linspace(99, 90, n)
assert np.allclose(feeder_jacobian(test_point, Vs_f, R_f),
                   numerical_jacobian(lambda x: feeder_mismatch(x, Vs_f, R_f, P_f), test_point), atol=1e-4)

v_f = solve_feeder(n, Vs_f, R_f, P_f)
assert np.abs(feeder_mismatch(v_f, Vs_f, R_f, P_f)).max() < 1e-6
assert np.all(np.diff(v_f) < 0), "voltage falls along the feeder"
v_all = np.concatenate([[Vs_f], v_f])
i_branch = (v_all[:-1] - v_all[1:]) / R_f
losses_f = (i_branch**2 * R_f).sum()
assert math.isclose(Vs_f * i_branch[0], n * P_f + losses_f, rel_tol=1e-9), "substation = loads + losses"
print(f"lowest voltage {v_f.min():.2f} V at bus {v_f.argmin() + 1}, losses {losses_f:.1f} W "
      f"({100 * losses_f / (n * P_f):.1f} % of load)")

**9. Interpretation**

Where along the feeder is the voltage lowest, and which segment carries the most current? The distribution operator must keep every bus above 90 % of nominal voltage (90 V) — does this feeder comply? Which two physical changes would you consider first, and how would each show up in your equations?

*Your answer here.*